# Advanced 01 — Advanced Authorization Models for Autonomous Agents

Hands-on course: ReBAC + ABAC + policy logic + task/time bounds for a claims-agent platform.


In [ ]:
from datetime import datetime,timedelta,timezone
import copy,itertools
import pandas as pd
import networkx as nx
NOW=datetime.now(timezone.utc)


## 1 — RBAC baseline

In [ ]:
ROLE_PERMS={
 "adjuster":{"claim.read","claim.update"},
 "manager":{"claim.read","claim.update","payment.approve"}
}
def rbac(role,action): return action in ROLE_PERMS.get(role,set())
rbac("adjuster","claim.update")


## 2 — Why RBAC explodes

In [ ]:
dimensions={"risk":3,"region":5,"assurance":3,"task_state":3,"data_class":4}
from math import prod
print("Potential role combinations:",prod(dimensions.values()))


## 3 — Build a ReBAC graph

In [ ]:
g=nx.MultiDiGraph()
g.add_edge("user:alice","claim:483",relation="assignee")
g.add_edge("agent:claims","user:alice",relation="acts_for")
g.add_edge("agent:claims","task:77",relation="assigned_to")
g.add_edge("task:77","claim:483",relation="can_update")
list(g.edges(data=True))


## 4 — ReBAC decision

In [ ]:
def has_edge(a,b,relation):
    return any(d["relation"]==relation for d in g.get_edge_data(a,b,default={}).values())
allowed=(has_edge("agent:claims","user:alice","acts_for") and
         has_edge("user:alice","claim:483","assignee") and
         has_edge("agent:claims","task:77","assigned_to") and
         has_edge("task:77","claim:483","can_update"))
allowed


## 5 — Delegation graph

In [ ]:
g.add_edge("user:alice","agent:claims",relation="delegates")
g.add_edge("agent:claims","agent:research",relation="delegates_read_only")


## 6 — Task-based authority

In [ ]:
task={"id":"task:77","agent":"agent:claims","active":True,
      "actions":{"claim.read","claim.update"},"resources":{"claim:483"},
      "expires":NOW+timedelta(minutes=30),"max_calls":5,"calls":0}


## 7 — Time and usage bounds

In [ ]:
def task_authorizes(t,action,resource,now):
    return (t["active"] and now<t["expires"] and t["calls"]<t["max_calls"]
            and action in t["actions"] and resource in t["resources"])
task_authorizes(task,"claim.update","claim:483",NOW)


## 8 — ABAC

In [ ]:
agent={"trust":0.92,"registered":True,"tenant":"acme"}
resource={"tenant":"acme","classification":"internal"}
context={"risk":"low","network":"corporate","purpose":"claims_processing"}
def abac(a,r,c):
    return (a["registered"] and a["trust"]>=0.8 and
            a["tenant"]==r["tenant"] and c["risk"]=="low")
abac(agent,resource,context)


## 9 — Attribute provenance

In [ ]:
attributes=pd.DataFrame([
 ["tenant","OIDC/server context","trusted"],
 ["workloadApproved","workload attestation","trusted"],
 ["risk","risk engine","trusted"],
 ["purpose","workflow/task service","trusted"],
 ["model_claimed_role","LLM output","untrusted"],
 ["tool_tenant_argument","model/tool input","untrusted"],
],columns=["attribute","source","trust"])
attributes


## 10 — Policy-based decision

In [ ]:
def policy(agent,resource,task,context):
    if agent["tenant"]!=resource["tenant"]: return "deny"
    if not agent["registered"]: return "deny"
    if not task_authorizes(task,"claim.update","claim:483",NOW): return "deny"
    if context["risk"]=="critical": return "deny"
    if context["risk"]=="high": return "step_up"
    if agent["trust"]<0.8: return "deny"
    return "allow"
policy(agent,resource,task,context)


## 11 — Cedar semantics

In [ ]:
def cedar_combine(permits,forbids):
    if any(forbids): return "deny"
    if any(permits): return "allow"
    return "deny"
print(cedar_combine([True],[False]))
print(cedar_combine([True],[True]))
print(cedar_combine([] ,[]))


## 12 — Decision constraints

In [ ]:
decision={"outcome":"allow","constraints":{"allowed_fields":{"status","notes"},"max_calls":2},"obligations":["audit"]}
attempt={"status":"reviewed"}
assert set(attempt).issubset(decision["constraints"]["allowed_fields"])


## 13 — Risk-adaptive policy

In [ ]:
for r in ["low","high","critical"]:
    c={**context,"risk":r}
    print(r,policy(agent,resource,task,c))


## 14 — Purpose-based policy

In [ ]:
def purpose_ok(c): return c["purpose"] in {"claims_processing","fraud_investigation"}
purpose_ok(context)


## 15 — Tenant substitution attack

In [ ]:
evil_resource={**resource,"tenant":"other"}
policy(agent,evil_resource,task,context)


## 16 — Attribute spoofing

In [ ]:
model_output={"tenant":"acme","risk":"low","trust":1.0}
print("Never use these as trusted security facts:",model_output)


## 17 — Freshness

In [ ]:
facts={"risk":{"value":"low","evaluated_at":NOW,"ttl":timedelta(seconds=30)}}
def fresh(f,now): return now < f["evaluated_at"]+f["ttl"]
fresh(facts["risk"],NOW+timedelta(seconds=20)),fresh(facts["risk"],NOW+timedelta(minutes=2))


## 18 — Cache-key design

In [ ]:
cache_key=("user:alice","agent:claims","workload:v7","task:77","claim.update","claim:483","acme","low","policy:v3")
cache_key


## 19 — Revocation

In [ ]:
revoked=copy.deepcopy(task);revoked["active"]=False
task_authorizes(revoked,"claim.update","claim:483",NOW)


## 20 — Decision table

In [ ]:
rows=[]
for trust,risk,tenant in itertools.product([0.5,0.9],["low","high","critical"],[True,False]):
    a={**agent,"trust":trust}
    r={**resource,"tenant":"acme" if tenant else "other"}
    c={**context,"risk":risk}
    rows.append([trust,risk,tenant,policy(a,r,task,c)])
pd.DataFrame(rows,columns=["trust","risk","tenant_match","decision"])


## 21 — Property: cross-tenant never allows

In [ ]:
table=pd.DataFrame(rows,columns=["trust","risk","tenant_match","decision"])
assert not ((table.tenant_match==False)&(table.decision=="allow")).any()


## 22 — Property: expired task never allows

In [ ]:
expired=copy.deepcopy(task);expired["expires"]=NOW-timedelta(seconds=1)
assert not task_authorizes(expired,"claim.update","claim:483",NOW)


## 23 — Mutation: remove tenant check

In [ ]:
def mutated_policy(agent,resource,task,context):
    # BUG: tenant check removed
    if not agent["registered"]: return "deny"
    if context["risk"]=="critical": return "deny"
    return "allow"
print("Mutation caught?",mutated_policy(agent,evil_resource,task,context)=="allow")


## 24 — Hybrid decision

In [ ]:
def hybrid(rebac_allowed,agent,resource,task,context):
    if not rebac_allowed:return "deny"
    return policy(agent,resource,task,context)
hybrid(allowed,agent,resource,task,context)


## 25 — OpenFGA lab

Use `policies/openfga/model.fga`.

Model:

```text
agent assigned to task
task permitted to tool
task permitted to claim
```

Then add contextual/task relationships and conditions for expiry or call budgets. Compare stored tuples, contextual tuples and conditions.


## 26 — OPA lab

Use `policies/opa/agent_authz.rego`.

Send trusted identity, workload, task, resource and risk data to OPA. Add:
- purpose restriction;
- data classification;
- approval requirement;
- max transaction value;
- explicit reason/obligation output.


## 27 — Cedar lab

Use `policies/cedar/agent_authz.cedar`.

Create Agent, Task, Claim and Tool entities. Exercise:
- default deny;
- permit;
- forbid override;
- missing context;
- cross-tenant request;
- critical-risk request.


## 28 — Capstone

Build a hybrid authorization service for an autonomous claims agent.

Requirements:

1. OpenFGA/ReBAC answers durable relationship questions.
2. Trusted runtime builds ABAC context.
3. OPA or Cedar evaluates contextual policy.
4. Task/time/call budgets bound autonomy.
5. High risk returns step-up.
6. Constraints are enforced by the PEP.
7. Revocation invalidates authority within a documented bound.
8. Cache keys include all security-relevant dimensions.
9. Cross-tenant access is impossible.
10. Mutation tests catch removed security predicates.
11. Decisions are explainable and auditable.
12. The model cannot provide trusted authorization attributes.


# Review questions

1. Why does RBAC produce role explosion for agents?
2. When is ReBAC preferable to ABAC?
3. When is ABAC preferable to ReBAC?
4. Why should tasks be authorization objects?
5. What are contextual tuples?
6. What is a conditional relationship?
7. Why does attribute provenance matter?
8. How should attribute freshness affect authorization?
9. What does Cedar's forbid semantics imply?
10. Why must decision constraints be enforced by a PEP?
11. How can risk adapt agent autonomy?
12. Why is purpose dangerous if model-generated?
13. What belongs in an authorization cache key?
14. How do revocation and consistency interact?
15. What does mutation testing tell you that positive tests do not?
16. Why is a hybrid model often better than one authorization paradigm?
